# Exploratory Data Analysis on a Cars Dataset
**Student:** KEERTHI H

This notebook explores the UCI Auto MPG dataset: data inspection, quality checks, conservative cleaning, descriptive statistics, visual EDA, outlier review, and evidence-based findings. No machine-learning model is built.

## 1. Setup and dataset source
The notebook downloads the public CSV from the seaborn-data repository. Internet access is required in Colab. The source is cited in the report and in the output below.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
df_raw = pd.read_csv(DATA_URL)

print("Dataset source:", DATA_URL)
print("Original shape:", df_raw.shape)
display(df_raw.head())


## 2. Understand the dataset
Inspect dimensions, columns, data types, and separate numerical from categorical features.

In [ ]:
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
print("\\nColumn names:")
print(df_raw.columns.tolist())
print("\\nData types:")
display(df_raw.dtypes.rename("dtype").to_frame())
numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_raw.select_dtypes(exclude=np.number).columns.tolist()
print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
display(df_raw.describe(include="all").T)


## 3. Data quality checks
Missing values, duplicate rows, and category consistency are checked before making changes.

In [ ]:
missing_before = df_raw.isna().sum().sort_values(ascending=False)
duplicates_before = int(df_raw.duplicated().sum())
print("Missing values by column:")
display(missing_before.to_frame("missing_count"))
print("Duplicate rows:", duplicates_before)
for col in categorical_cols:
    print(f"\\nUnique values in {col} (first 20):")
    print(df_raw[col].dropna().astype(str).str.strip().unique()[:20])


## 4. Cleaning decisions
Whitespace is stripped from text fields, exact duplicate rows are removed, and rows missing the target measurement `mpg` are excluded because fuel economy is central to the analysis. Missing values in other columns are retained and handled per analysis rather than silently imputed.

In [ ]:
df = df_raw.copy()
for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip()

df = df.drop_duplicates()
df = df.dropna(subset=["mpg"]).copy()

# Treat horsepower as numeric; invalid text becomes missing rather than a fabricated value.
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce").astype("Int64")
df["cylinders"] = pd.to_numeric(df["cylinders"], errors="coerce").astype("Int64")
df["origin"] = df["origin"].astype("category")

print(f"Rows before cleaning: {len(df_raw):,}")
print(f"Rows after cleaning:  {len(df):,}")
print(f"Columns before/after: {df_raw.shape[1]} / {df.shape[1]}")
print("Rows removed:", len(df_raw) - len(df))
print("Missing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows after cleaning:", int(df.duplicated().sum()))
display(df.head())


## 5. Descriptive analysis
Summary statistics describe numeric distributions and category frequencies.

In [ ]:
display(df.describe(include="number").T.round(2))
for col in ["cylinders", "origin", "model_year"]:
    if col in df.columns:
        print(f"\\nFrequency table: {col}")
        display(df[col].value_counts(dropna=False).rename_axis(col).to_frame("count"))


## 6. Visual exploratory data analysis
Each chart includes a short, data-driven observation. Charts are generated only when the notebook runs against the downloaded dataset.

In [ ]:
# Chart 1: fuel economy distribution
plt.figure(figsize=(8, 4.5))
sns.histplot(data=df, x="mpg", bins=24, kde=True)
plt.title("Distribution of Fuel Economy (MPG)")
plt.xlabel("Miles per gallon (MPG)")
plt.ylabel("Number of cars")
plt.tight_layout()
plt.show()
print(f"Observation: MPG ranges from {df['mpg'].min():.1f} to {df['mpg'].max():.1f}; median is {df['mpg'].median():.1f}.")


In [ ]:
# Chart 2: car counts by cylinder count
plt.figure(figsize=(8, 4.5))
sns.countplot(data=df, x="cylinders", order=sorted(df["cylinders"].dropna().unique()))
plt.title("Cars by Number of Cylinders")
plt.xlabel("Cylinders")
plt.ylabel("Number of cars")
plt.tight_layout()
plt.show()
print("Observation: The tallest bar identifies the most common cylinder configuration in this dataset.")


In [ ]:
# Chart 3: fuel economy by cylinder count
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x="cylinders", y="mpg")
plt.title("Fuel Economy by Cylinder Count")
plt.xlabel("Cylinders")
plt.ylabel("Miles per gallon (MPG)")
plt.tight_layout()
plt.show()
means_cyl = df.groupby("cylinders", observed=True)["mpg"].median().sort_index()
print("Observation: Median MPG by cylinder count:")
display(means_cyl.round(2).to_frame("median_mpg"))


In [ ]:
# Chart 4: vehicle weight and fuel economy
plt.figure(figsize=(8, 4.5))
sns.scatterplot(data=df, x="weight", y="mpg", hue="origin", alpha=0.8)
plt.title("Vehicle Weight vs Fuel Economy")
plt.xlabel("Weight (dataset units)")
plt.ylabel("Miles per gallon (MPG)")
plt.tight_layout()
plt.show()
corr_weight = df[["weight", "mpg"]].corr().iloc[0, 1]
print(f"Observation: Pearson correlation between weight and MPG is {corr_weight:.2f}; correlation describes association, not causation.")


In [ ]:
# Chart 5: horsepower and fuel economy
plt.figure(figsize=(8, 4.5))
sns.regplot(data=df, x="horsepower", y="mpg", scatter_kws={"alpha": 0.55}, line_kws={"color": "black"})
plt.title("Horsepower vs Fuel Economy")
plt.xlabel("Horsepower")
plt.ylabel("Miles per gallon (MPG)")
plt.tight_layout()
plt.show()
hp_corr = df[["horsepower", "mpg"]].corr().iloc[0, 1]
print(f"Observation: Pearson correlation between horsepower and MPG is {hp_corr:.2f}, using rows with both values present.")


In [ ]:
# Chart 6: fuel economy across model years
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x="model_year", y="mpg", color="lightsteelblue")
plt.title("Fuel Economy by Model Year")
plt.xlabel("Model year (two-digit code in source)")
plt.ylabel("Miles per gallon (MPG)")
plt.tight_layout()
plt.show()
year_median = df.groupby("model_year", observed=True)["mpg"].median()
print(f"Observation: Median MPG changes from {year_median.iloc[0]:.1f} in the earliest represented model year to {year_median.iloc[-1]:.1f} in the latest.")


In [ ]:
# Chart 7: correlation heatmap
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=False)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()
print("Observation: The heatmap highlights pairs of numeric variables with stronger positive or negative linear associations.")


## 7. Outlier review
The IQR rule flags potential outliers for review. A flag is not an automatic reason to delete a record; unusual vehicles may be legitimate.

In [ ]:
outlier_rows = []
for col in ["mpg", "displacement", "horsepower", "weight", "acceleration"]:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((s < low) | (s > high)).sum())
    outlier_rows.append({"feature": col, "lower_fence": low, "upper_fence": high, "potential_outliers": count})
display(pd.DataFrame(outlier_rows).round(2))
print("Decision: potential outliers are reported, not removed automatically; investigate source records before changing them.")


## 8. Five or more evidence-based findings
The statements below are calculated from the cleaned data, so the values update if the source dataset changes.

In [ ]:
findings = []
findings.append(f"Fuel economy: median MPG is {df['mpg'].median():.2f}, with a range of {df['mpg'].min():.1f}–{df['mpg'].max():.1f}.")
findings.append(f"Weight relationship: weight–MPG correlation is {corr_weight:.2f}.")
findings.append(f"Horsepower relationship: horsepower–MPG correlation is {hp_corr:.2f}.")
if len(means_cyl.dropna()):
    findings.append(f"Cylinder comparison: the highest median MPG is in the {means_cyl.idxmax()}-cylinder group ({means_cyl.max():.2f} MPG).")
if len(year_median.dropna()):
    findings.append(f"Model-year pattern: median MPG is {year_median.iloc[0]:.2f} in the earliest represented year and {year_median.iloc[-1]:.2f} in the latest.")
origin_median = df.groupby("origin", observed=True)["mpg"].median().sort_values(ascending=False)
if len(origin_median):
    findings.append(f"Origin comparison: the highest median MPG among origin codes is {origin_median.index[0]} ({origin_median.iloc[0]:.2f} MPG).")
for i, item in enumerate(findings, 1):
    print(f"{i}. {item}")


## 9. Limitations and conclusion
- The dataset is historical and may not represent current vehicles or current fuel-economy standards.
- The `origin` field is encoded as a category/code; its interpretation should follow the dataset documentation.
- Missing horsepower values can reduce the sample used in horsepower comparisons.
- Correlation does not prove causation, and observational data may contain confounding factors.
- Outliers are flagged for review rather than automatically removed.

**Conclusion:** This workflow demonstrates a complete, reproducible EDA process: inspect, validate, clean conservatively, summarize, visualize, and communicate findings. Review the outputs and be prepared to explain each decision.